# LCA Lasers4MaaS: workflow assembly

Adds up the material, pre-joining, joining and post-joining impacts of each product, for each joining technology.

Run `joining_task.ipynb` and `pre_post.ipynb` first. This notebook reads their result files in `results/`.
The calculation code is in the `l4m_lca` package. This notebook holds the parameters and shows the results.

In [ ]:
import os

import pandas as pd
from IPython.display import display

from l4m_lca import plots
from l4m_lca.impacts import BackgroundImpacts
from l4m_lca.constants import BACKGROUNDS
from l4m_lca.lcia import COST_COLUMN
from l4m_lca.project import DEFAULT_INDICATORS, ef31_methods, find_activity, method_units, open_project
from l4m_lca.workflow import (STAGE_TITLES, cumulative_regional, material_table, read_results, unit_of,
                              workflow_table, write_json)

## Parameters

In [ ]:
# Background database: "ecoinvent" or "bafu". The environment variable L4M_BACKGROUND overrides it.
BACKGROUND = os.environ.get("L4M_BACKGROUND", "ecoinvent")
PROFILE = BACKGROUNDS[BACKGROUND]
PROJECT_NAME = PROFILE["project"]
BACKGROUND_DATABASE = PROFILE["database"]
RESULTS_DIR = PROFILE["results"]

# Result files of the other notebooks
JOINING_RESULTS = f"{RESULTS_DIR}/State_of_the_Art_Dynamic_Beam_Shaping_LCIA_results.json"
PREPOST_RESULTS = f"{RESULTS_DIR}/Chemical_Pre_and_Post_processes_Mechanical_Pre_and_Post_processes_LCIA_results.json"
REGIONAL_JOINING = f"{RESULTS_DIR}/regional_joining.csv"
REGIONAL_PREPOST = f"{RESULTS_DIR}/regional_prepost.csv"
MATERIAL_RESULTS = f"{RESULTS_DIR}/Material_production_LCIA_results.json"

INDICATORS = DEFAULT_INDICATORS
DEFAULT_IMPACT = "climate change: total (EF v3.1 - IPCC 2021)"
SCENARIO = "amount"           # "minimum", "amount" or "maximum"
MATERIAL_CASE = "prototype"   # "prototype" or "production": selects the part weight

# Material markets (name, location) of each background; keys = PRODUCT_WORKFLOWS[...]["material"]
MATERIAL_MARKETS = {
    "ecoinvent": {"aluminium": ("market for aluminium, wrought alloy", "GLO"),
                  "steel": ("market for steel, chromium steel 18/8", "GLO")},
    "bafu": {"aluminium": ("Aluminium, production mix, wrought alloy, at plant", "RER"),
             "steel": ("Chromium steel 18/8, at plant", "RER")},
}

In [ ]:
# Product-specific workflow: material, enabled pre- and post-joining steps, prototype/production parameters
PRODUCT_WORKFLOWS = {
    "Heat exchanger": {
        "material": "aluminium",
        "pre_joining": {"alkaline_degreasing": True, "mechanical_treatment": True, "machining": False},
        "post_joining": {"chemical_etching": True, "conversion_coating": True, "grinding_polishing": True},
        "process_parameters": {
            "prototype": {
                "surface_area_m2": {"min": 0.5, "max": 2.0},
                "part_weight_kg": {"min": 1.0, "max": 5.0},
                "joint_length_m": {"min": 5.0, "max": 15.0},
            },
            "production": {
                "surface_area_m2": {"min": 1.0, "max": 5.0},
                "part_weight_kg": {"min": 5.0, "max": 20.0},
                "joint_length_m": {"min": 10.0, "max": 30.0},
            },
        },
    },
    "Fuel tank": {
        "material": "steel",
        "pre_joining": {
            "alkaline_degreasing": True,
            "mechanical_treatment": True,  # oxides/scale likely present
            "machining": True,             # fuel tanks require tight tolerances
        },
        "post_joining": {"pickling": True, "passivation": True, "grinding_polishing": True},
        "process_parameters": {
            "prototype": {
                "surface_area_m2": {"min": 2.0, "max": 10.0},
                "part_weight_kg": {"min": 10.0, "max": 50.0},
                "joint_length_m": {"min": 2.0, "max": 2.2},
            },
            "production": {
                "surface_area_m2": {"min": 5.0, "max": 20.0},
                "part_weight_kg": {"min": 50.0, "max": 200.0},
                "joint_length_m": {"min": 2.0, "max": 2.2},
            },
        },
    },
    "Heterogeneous pipe": {
        "material": "steel",
        "pre_joining": {
            "alkaline_degreasing": True,
            "mechanical_treatment": True,  # oxides/scale present on pipes
            "machining": False,
        },
        "post_joining": {"pickling": True, "passivation": True, "grinding_polishing": True},
        "process_parameters": {   # no part weight yet: the material stage is 0
            "pipe_diameter_mm": {"min": 50, "max": 200},
            "pipe_length_m": {"min": 1.0, "max": 10.0},
            "joint_length_m": {"min": 0.5, "max": 3.0},
        },
    },
    "Freeze Dryers Shelf": {
        "material": "steel",
        "pre_joining": {"alkaline_degreasing": True, "mechanical_treatment": True, "machining": False},
        "post_joining": {"pickling": True, "passivation": True, "grinding_polishing": False},
        "process_parameters": {
            "prototype": {
                "surface_area_m2": {"min": 1.0, "max": 3.0},
                "part_weight_kg": {"min": 20.0, "max": 100.0},
                "joint_length_m": {"min": 5.0, "max": 20.0},
            },
            "production": {
                "surface_area_m2": {"min": 3.0, "max": 10.0},
                "part_weight_kg": {"min": 100.0, "max": 300.0},
                "joint_length_m": {"min": 10.0, "max": 40.0},
            },
        },
    },
    "Battery cooling plate": {
        "material": "aluminium",
        "pre_joining": {"alkaline_degreasing": True, "mechanical_treatment": True, "machining": False},
        "post_joining": {"chemical_etching": True, "conversion_coating": False, "grinding_polishing": True},
        "process_parameters": {
            "prototype": {
                "surface_area_m2": {"min": 0.1, "max": 0.5},
                "part_weight_kg": {"min": 1.26, "max": 14.38},
                "joint_length_m": {"min": 0.5, "max": 2.0},
            },
            "production": {
                "surface_area_m2": {"min": 0.5, "max": 2.0},
                "part_weight_kg": {"min": 14.38, "max": 57.52},
                "joint_length_m": {"min": 2.0, "max": 8.0},
            },
        },
    },
}

## Material production LCIA (per kg)

EF 3.1 impacts of 1 kg of each material market. The material stage of a product = impact per kg x mean part weight of `MATERIAL_CASE`.

In [ ]:
open_project(PROJECT_NAME, BACKGROUND_DATABASE)
methods = ef31_methods(INDICATORS)
units = method_units(methods)
markets = {m: find_activity(BACKGROUND_DATABASE, *spec) for m, spec in MATERIAL_MARKETS[BACKGROUND].items()}
impacts = BackgroundImpacts(methods, next(iter(markets.values())))

material_df = material_table(markets, impacts, units, BACKGROUND_DATABASE)
write_json(material_df, MATERIAL_RESULTS)
material_df.set_index("Key").T

## Workflow assembly

In [ ]:
joining_df = read_results(JOINING_RESULTS)
prepost_df = read_results(PREPOST_RESULTS)
workflow_df = workflow_table(PRODUCT_WORKFLOWS, joining_df, prepost_df, material_df, MATERIAL_CASE, SCENARIO)
workflow_df[workflow_df["indicator"].str.startswith(DEFAULT_IMPACT)].pivot_table(
    index=["product", "technology"], columns=["stage", "step"], values="value", sort=False)

In [ ]:
display(plots.workflow_bars(workflow_df, STAGE_TITLES, DEFAULT_IMPACT, SCENARIO))

## Cumulative regional analysis: EU electricity markets

Adds up the regional impacts of the enabled pre-joining steps, the joining process and the enabled post-joining steps of each product, with the electricity market of each member state.
The material stage is not included. Pre/post steps and joining use the `SCENARIO` scenario.

In [ ]:
reg_joining = pd.read_csv(REGIONAL_JOINING)
reg_prepost = pd.read_csv(REGIONAL_PREPOST)
cumulative_df = cumulative_regional(PRODUCT_WORKFLOWS, reg_joining, reg_prepost, SCENARIO)

regional_units = {**units, COST_COLUMN: "EUR"}
for col in joining_df.columns:   # units from the result column names, e.g. "acidification [mol H+-Eq]"
    if "[" in col:
        regional_units.setdefault(col.split(" [")[0], unit_of(col))

display(plots.regional_explorer(
    cumulative_df,
    filters=[("Product", "product", list(PRODUCT_WORKFLOWS), None),
             ("Technology", "technology", sorted(cumulative_df["technology"].unique()), None)],
    units=regional_units,
    title=lambda f: f"{f['product']}, {f['technology']}: cumulative pre + joining + post",
    scatter_on_top=True, height=1400, width=1400,
))